# Module 6.1 — Contextual Compression

Problem: retrieved chunks contain **irrelevant sentences** that waste context window and confuse the LLM.

Solution: compress retrieved documents to only the relevant parts.

| Compressor | Mechanism | Speed |
|---|---|---|
| `LLMChainExtractor` | LLM extracts relevant sentences | Slow |
| `EmbeddingsFilter` | Drops chunks below sim threshold | Fast |
| `DocumentCompressorPipeline` | Chain multiple compressors | Configurable |

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import (
    LLMChainExtractor,
    EmbeddingsFilter,
    DocumentCompressorPipeline,
)
from langchain.schema import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Verbose docs — contain both relevant and irrelevant sentences
docs = [
    Document(page_content=(
        "Python was created by Guido van Rossum. "
        "The language emphasises readability and simplicity. "
        "Python supports multiple programming paradigms. "
        "The Eiffel Tower is 330 metres tall. "
        "Python is widely used in data science and machine learning."
    )),
    Document(page_content=(
        "Machine learning is a subset of artificial intelligence. "
        "It enables computers to learn from data without explicit programming. "
        "The Sahara Desert is the world's largest hot desert. "
        "Supervised learning uses labelled training data."
    )),
]

vs        = Chroma.from_documents(docs, embeddings, collection_name="compress_demo")
retriever = vs.as_retriever(search_kwargs={"k": 2})
query     = "What is Python used for?"

# ── Baseline: no compression ──────────────────────────────────────────────────
baseline = retriever.invoke(query)
print("BASELINE (no compression):")
for d in baseline:
    print(f"  {d.page_content}\n")

# ── LLMChainExtractor ─────────────────────────────────────────────────────────
llm_compressor = LLMChainExtractor.from_llm(llm)
llm_retriever  = ContextualCompressionRetriever(
    base_compressor=llm_compressor, base_retriever=retriever
)
llm_results = llm_retriever.invoke(query)
print("LLMChainExtractor:")
for d in llm_results:
    print(f"  {d.page_content}\n")

# ── EmbeddingsFilter (fast) ───────────────────────────────────────────────────
emb_filter     = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.76)
emb_retriever  = ContextualCompressionRetriever(
    base_compressor=emb_filter, base_retriever=retriever
)
emb_results = emb_retriever.invoke(query)
print("EmbeddingsFilter:")
for d in emb_results:
    print(f"  {d.page_content}\n")

# ── Pipeline: EmbeddingsFilter → LLMChainExtractor ───────────────────────────
pipeline   = DocumentCompressorPipeline(transformers=[emb_filter, llm_compressor])
pipe_retr  = ContextualCompressionRetriever(
    base_compressor=pipeline, base_retriever=retriever
)
pipe_res = pipe_retr.invoke(query)
print("Pipeline (filter → extract):")
for d in pipe_res:
    print(f"  {d.page_content}\n")
